In [1]:
# Import libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import joblib

In [ ]:
# Import data from csv
data = pd.read_csv('../data/cleaned_data.csv')

# Extract findings and convert to list
findings = data['findings'].tolist()

findings[:5]

['The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no [REDACTED] of a pleural effusion. There is no evidence of pneumothorax.',
 'Borderline cardiomegaly. Midline sternotomy [REDACTED]. Enlarged pulmonary arteries. Clear lungs. Inferior [REDACTED] [REDACTED] [REDACTED].',
 'There are diffuse bilateral interstitial and alveolar opacities consistent with chronic obstructive lung disease and bullous emphysema. There are irregular opacities in the left lung apex, that could represent a cavitary lesion in the left lung apex.There are streaky opacities in the right upper lobe, [REDACTED] scarring. The cardiomediastinal silhouette is normal in size and contour. There is no pneumothorax or large pleural effusion.',
 'The cardiomediastinal silhouette and pulmonary vasculature are within normal limits. There is no pneumothorax or pleural effusion. There are no focal areas of consolidation. Cholecystectom

In [3]:
# Create vectorizer
# Ignore terms that appear more than 80%
# Define custom stopwords for the vectorizer
custom_stopwords = ['is', 'are', 'the', 'of', 'or', 'and', 'without', 'there', 'redacted', 'no', 'in']
vectorizer = TfidfVectorizer(max_df=.8, ngram_range=(1,2), stop_words=custom_stopwords)

# Fit/transform findings
tfidf_matrix = vectorizer.fit_transform(findings)

In [4]:
# Retrieve 100 top words
count_vectorizer = CountVectorizer(max_df=.9, min_df=2, stop_words=custom_stopwords)
count_matrix = count_vectorizer.fit_transform(findings)


word_counts = count_matrix.sum(axis=0).A1
words = count_vectorizer.get_feature_names_out()
word_freq = sorted(zip(words,word_counts), key=lambda x: x[1], reverse=True)
top_words = list(word for word, _ in word_freq[:100])
top_words

['normal',
 'pleural',
 'pneumothorax',
 'effusion',
 'heart',
 'size',
 'focal',
 'lungs',
 'within',
 'limits',
 'clear',
 'right',
 'pulmonary',
 'consolidation',
 'left',
 'silhouette',
 'airspace',
 'mediastinal',
 'lung',
 'cardiomediastinal',
 'stable',
 'with',
 'spine',
 'changes',
 'thoracic',
 'disease',
 'acute',
 'degenerative',
 'mild',
 'contours',
 'unremarkable',
 'structures',
 'calcified',
 'mediastinum',
 'seen',
 'bony',
 'large',
 'contour',
 'opacity',
 'cardiac',
 'upper',
 'appear',
 'to',
 'effusions',
 'opacities',
 'vascularity',
 'noted',
 'intact',
 'aorta',
 'osseous',
 'lobe',
 'for',
 'lateral',
 'chest',
 'visualized',
 'evidence',
 'abnormality',
 'identified',
 'lower',
 'atelectasis',
 'present',
 'volumes',
 'vasculature',
 'prior',
 'air',
 'on',
 'low',
 'grossly',
 'unchanged',
 'granuloma',
 'enlarged',
 'infiltrate',
 'bilateral',
 'base',
 'appearance',
 'scarring',
 'small',
 'rib',
 'bilaterally',
 'space',
 'findings',
 'cardiomegaly',
 'e

In [5]:
# Test vectorizer on given examples and few new inputs
test1 = 'The trachea is midline. The cardiomediastinal silhouette is normal. The lungs are clear, without evidence of acute infiltrate or effusion. There is no pneumothorax. The visualized bony structures reveal no acute abnormalities.'
test2 = 'The lungs are clear. Heart size and mediastinal contours are normal. No osseous abnormalities.'
test3 = 'AP and lateral views were obtained. Bibasilar atelectasis and small left-sided pleural effusion. Stable cardiomegaly. No pneumothorax. Mild pulmonary vascular congestion'
test4 = 'I like apples, oranges, and bananas. Fruits are healthy, and delicious.'
test5 = 'This recepie requires flour, sugar and eggs. It is very quick and easy.'
test6 = 'Todays weather is going to be rainy, but do not worry, it will clear up by the evening.'


# Check similarity score on input
def check_similarity(input):
    input_vec = vectorizer.transform([input])
    similarity = cosine_similarity(input_vec, tfidf_matrix).flatten()
    return similarity.max()

# Check if vocab is present in the input
def vocab_presence(input, vocab=top_words):
    if [i for i in vocab if i in input.lower()]:
        return True
    return False

# Check input validity
def validate_input(input):
    if vocab_presence(input):
        if check_similarity(input) > 0.2:
            return True
        else:
            return 'Input not similar enough'
    else:
        return 'Input not valid'

In [6]:
# Check input validity for test cases
for test in [test1, test2, test3, test4, test5, test6]:
    print(f'validate_input(test): {validate_input(test)}')

validate_input(test): True
validate_input(test): True
validate_input(test): True
validate_input(test): Input not valid
validate_input(test): Input not valid
validate_input(test): Input not similar enough


In [7]:
# Check similarity score for tests cases
for test in [test1, test2, test3, test4, test5, test6]:
    print(check_similarity(test))

1.0
0.4842728647488953
0.2890798822542008
0.19925377431000144
0.12465314812124546
0.17806688082884906


In [ ]:
# Path
path = '../model/tfidf/'

# Save trained model 
joblib.dump(vectorizer, path + 'tfidf_vectorizer.joblib')
joblib.dump(tfidf_matrix, path + 'tfidf_matrix.joblib')
joblib.dump(top_words, path + 'top_words.joblib')

print('Models saved')

Models saved
